# Who Wrote It?

CMPSC 448 solo project. In this notebook we will investigate **RQ1** and **RQ2** using model family labels for **Claude**, **GPT**, and **Qwen** responses from the `lmarena-ai/arena-expert-5k` dataset.

This initial setup only loads and inspects the data.

In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("lmarena-ai/arena-expert-5k")

In [2]:
# Available splits and their sizes
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'model_a', 'model_b', 'winner', 'conversation_a', 'conversation_b', 'evaluation_order', 'full_conversation', 'language', 'occupational_tags'],
        num_rows: 5128
    })
})

In [3]:
# Column names and row count of the train split
train = ds["train"]
print("Columns:", train.column_names)
print("Rows:", train.num_rows)

Columns: ['id', 'model_a', 'model_b', 'winner', 'conversation_a', 'conversation_b', 'evaluation_order', 'full_conversation', 'language', 'occupational_tags']
Rows: 5128


In [4]:
# One example row from the train split
train[0]

{'id': 'f744f0c7-ed1e-43dc-b9b5-d765d2d4ce01',
 'model_a': 'claude-3-5-sonnet-20241022',
 'model_b': 'gemma-3-27b-it',
 'winner': 'both_bad',
 'conversation_a': '[{\'role\': \'user\', \'content\': array([{\'type\': \'text\', \'text\': \'一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000,S=100000）\', \'image\': None, \'mimeType\': None}],\n       dtype=object)}\n {\'role\': \'assistant\', \'content\': array([{\'type\': \'text\', \'text\': \'让我一步步思考这个问题：\\n\\n1) 这是一个组合数学问题，本质上是求将S分成n个非负整数且要求单调不降的方案数\\n\\n2) 这个问题可以转化为将S个球放入n个盒子中，要求每个盒子的球数不少于前一个盒子的球数\\n\\n3) 这个问题有一个经典解法，可以转化为"隔板法"：\\n   - 将S个球排成一行\\n   - 需要放n-1个隔板来分成n份\\n   - 由于要求单调不降，所以每个隔板只能放在"相等或上升"的位置\\n\\n4) 这可以进一步转化为：在S+n-1个位置中选择n-1个位置放隔板\\n   - 总共有S个球和n-1个隔板，共S+n-1个位置\\n   - 需要选择n-1个位置放隔板\\n\\n5) 因此，答案就是组合数C(S+n-1, n-1)\\n\\n6) 代入数据：n=1000, S=100000\\n   答案 = C(100999, 999)\\n\\n这个组合数会是一个非常大的数。由于数值太大，如果需要具体计算可能需要用到大数运算或者取模运算。\\n\\n需要注意的是，这个解法是基于经典的"隔板法"或"插板法"，是一个非常优雅的组合数学解决方案。\', \'image\': None, \'mimeType\': None}],\n       dtype=

## Dataset verification: field types, languages, and models

Before extracting any examples, check how the conversation fields are stored, which languages appear, and which models appear. Nothing is parsed or filtered here.

In [5]:
# Python types of the conversation fields in the first row, with a short preview
CONV_COLS = ["conversation_a", "conversation_b", "full_conversation"]
row = train[0]
for col in CONV_COLS:
    print(f"{col}: {type(row[col]).__name__}")
    print("  preview:", repr(row[col])[:200])

conversation_a: str
  preview: '[{\'role\': \'user\', \'content\': array([{\'type\': \'text\', \'text\': \'一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000,S=100000）\', \'image\': None, \'mimeType\': None}],\n       dtype=object)
conversation_b: str
  preview: '[{\'role\': \'user\', \'content\': array([{\'type\': \'text\', \'text\': \'一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000,S=100000）\', \'image\': None, \'mimeType\': None}],\n       dtype=object)
full_conversation: str
  preview: '[{\'user\': {\'role\': \'user\', \'content\': array([{\'type\': \'text\', \'text\': \'一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000,S=100000）\', \'image\': None, \'mimeType\': None}],\n       dt


In [6]:
from collections import Counter

# Are the types consistent across all rows? Count missing (None) and empty values
for col in CONV_COLS:
    values = list(train[col])
    types = Counter(type(v).__name__ for v in values)
    missing = sum(v is None for v in values)
    empty = sum(v is not None and len(v) == 0 for v in values)
    print(f"{col}: types={dict(types)}, missing={missing}, empty={empty}")

conversation_a: types={'str': 5128}, missing=0, empty=0


conversation_b: types={'str': 5128}, missing=0, empty=0


full_conversation: types={'str': 5128}, missing=0, empty=0


In [7]:
# Rows per language value
language_counts = Counter(train["language"])
print("Distinct languages:", len(language_counts))
language_counts.most_common()

Distinct languages: 35


[('en', 3032),
 ('zh', 548),
 ('und', 340),
 ('ru', 279),
 ('ja', 277),
 ('pl', 141),
 ('ko', 121),
 ('de', 101),
 ('fr', 52),
 ('fa', 43),
 ('es', 42),
 ('pt', 31),
 ('it', 29),
 ('zh-Hant', 28),
 ('ar', 13),
 ('vi', 6),
 ('<err>', 5),
 ('el', 4),
 ('iw', 4),
 ('tr', 4),
 ('id', 3),
 ('bg', 3),
 ('cs', 3),
 ('uk', 3),
 ('ro', 3),
 ('la', 2),
 ('sk', 2),
 ('sv', 2),
 ('co', 1),
 ('bn', 1),
 ('kha', 1),
 ('gu', 1),
 ('hu', 1),
 ('no', 1),
 ('nl', 1)]

In [8]:
# Model appearances across model_a and model_b combined.
# These count source rows, not usable or unique responses.
model_counts = Counter(list(train["model_a"]) + list(train["model_b"]))
print("Distinct models:", len(model_counts))
model_counts.most_common()

Distinct models: 105


[('claude-opus-4-20250514', 246),
 ('qwen3-235b-a22b-no-thinking', 244),
 ('claude-sonnet-4-20250514', 243),
 ('o3-2025-04-16', 238),
 ('o3-mini', 230),
 ('gpt-4.1-mini-2025-04-14', 230),
 ('o4-mini-2025-04-16', 218),
 ('gpt-4.1-2025-04-14', 218),
 ('deepseek-v3-0324', 211),
 ('claude-3-5-haiku-20241022', 210),
 ('mistral-medium-2505', 208),
 ('chatgpt-4o-latest-20250326-old', 206),
 ('command-a-03-2025', 204),
 ('gemini-2.5-flash-lite-preview-06-17-thinking', 197),
 ('claude-3-5-sonnet-20241022', 194),
 ('llama-4-maverick-17b-128e-instruct', 192),
 ('claude-opus-4-20250514-thinking-16k', 183),
 ('gemma-3-27b-it', 183),
 ('claude-opus-4-1-20250805', 179),
 ('llama-4-scout-17b-16e-instruct', 173),
 ('claude-sonnet-4-20250514-thinking-32k', 171),
 ('kimi-k2-0711-preview', 171),
 ('claude-3-7-sonnet-20250219-thinking-32k', 169),
 ('claude-3-7-sonnet-20250219', 169),
 ('qwen3-235b-a22b-instruct-2507', 162),
 ('grok-3-mini-beta', 159),
 ('mistral-small-3.1-24b-instruct-2503', 155),
 ('llama

## Dataset verification: conversation parsing and attribution (samples only)

Three English sample rows (one each with `evaluation_order` 1, 2, 3) are used to check how conversations are stored and which model wrote each response. Findings from three rows are not rules for the whole dataset.

In [9]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

# Schema of the already-cached source parquet file (local_files_only: no download)
parquet_path = hf_hub_download("lmarena-ai/arena-expert-5k", "data/train-00000-of-00001.parquet",
                               repo_type="dataset", local_files_only=True)
schema = pq.read_schema(parquet_path)
for col in ["model_a", "model_b", "conversation_a", "conversation_b", "evaluation_order", "full_conversation"]:
    print(f"{col}: {schema.field(col).type}")

model_a: string
model_b: string
conversation_a: string
conversation_b: string
evaluation_order: int32
full_conversation: string


In [10]:
# First English row for each evaluation_order 1, 2, 3
langs, orders = train["language"], train["evaluation_order"]
sample_idx = [next(i for i in range(len(train)) if langs[i] == "en" and orders[i] == k) for k in (1, 2, 3)]
for i in sample_idx:
    r = train[i]
    print(i, r["id"], "| order", r["evaluation_order"], "|", r["model_a"], "vs", r["model_b"])

2 63e84dcc-7ddd-42ff-a153-e0d961cc0d03 | order 1 | minimax-m1 vs deepseek-r1-0528
7 9829f6ea-28fb-4568-bdee-62c1350d2eb6 | order 2 | o3-2025-04-16 vs grok-3-mini-beta
53 b8d7e4f8-4dd8-484e-b2df-bdd473a714af | order 3 | o3-2025-04-16 vs command-a-03-2025


The conversation strings look like Python reprs of numpy object arrays. Removing the `array(` wrapper with text replacement is unsafe, because the same characters could appear inside prompts or responses. Instead, Python's `tokenize` module splits the string into tokens. String literals stay single tokens, so their contents are never touched. The parser:

1. drops the `array(` and `, dtype=object)` tokens, and turns the exact empty form `array([], dtype=float64)` into `[]`;
2. inserts the missing comma between `}` and `{` (numpy prints object arrays without commas);
3. rejects any other name token (anything other than `None`/`True`/`False`) and any `...` truncation marker, instead of repairing it;
4. hands the result to `ast.literal_eval`, which only accepts literals and never executes code.

A check then confirms that every string in the parsed result equals the original string token, decoded on its own.

In [11]:
import ast, io, tokenize

SKIP = (tokenize.NL, tokenize.NEWLINE, tokenize.ENDMARKER)

def parse_conversation(s):
    """Parse one numpy-repr conversation string without executing it."""
    toks = [t for t in tokenize.generate_tokens(io.StringIO(s).readline) if t.type not in SKIP]
    out, i = [], 0
    while i < len(toks):
        t = toks[i]
        if [x.string for x in toks[i:i + 9]] == ["array", "(", "[", "]", ",", "dtype", "=", "float64", ")"]:
            out += ["[", "]"]; i += 9; continue                # exact empty array -> []
        if t.string == "array" and toks[i + 1].string == "(":
            i += 2; continue                                   # drop "array("
        if [x.string for x in toks[i:i + 5]] == [",", "dtype", "=", "object", ")"]:
            i += 5; continue                                   # drop ", dtype=object)"
        if t.type == tokenize.NAME and t.string not in ("None", "True", "False"):
            raise ValueError(f"unexpected name token: {t.string}")
        if t.string == "...":
            raise ValueError("truncation marker found")
        if t.string == "{" and out and out[-1] == "}":
            out.append(",")                                    # missing separator
        out.append(t.string)
        i += 1
    parsed = ast.literal_eval(" ".join(out))
    # Faithfulness check: parsed strings must equal the original string tokens, in order
    originals = [ast.literal_eval(t.string) for t in toks if t.type == tokenize.STRING]
    assert list(all_strings(parsed)) == originals, "text changed during parsing"
    return parsed

def all_strings(o):
    """Yield every string (keys and values) in document order."""
    if isinstance(o, dict):
        for k, v in o.items():
            yield k; yield from all_strings(v)
    elif isinstance(o, list):
        for v in o: yield from all_strings(v)
    elif isinstance(o, str):
        yield o

In [12]:
# Parse the three samples and show their structure
def text_of(msg):
    return msg["content"][0]["text"]

for i in sample_idx:
    r = train[i]
    conv_a, conv_b = parse_conversation(r["conversation_a"]), parse_conversation(r["conversation_b"])
    full = parse_conversation(r["full_conversation"])
    print(f"row {i} (order {r['evaluation_order']}):")
    print("  conversation_a roles:", [m["role"] for m in conv_a], "| conversation_b roles:", [m["role"] for m in conv_b])
    print("  conversation_b extra keys:", sorted(set(conv_b[0]) - set(conv_a[0])))
    print("  full_conversation turns:", len(full), "| keys per turn:", list(full[0].keys()))
    last = full[-1]
    print("  last turn matches conversation_a/b:",
          text_of(last["user"]) == text_of(conv_a[0]) == text_of(conv_b[0]),
          text_of(last["model_a"]) == text_of(conv_a[1]), text_of(last["model_b"]) == text_of(conv_b[1]))

row 2 (order 1):
  conversation_a roles: ['user', 'assistant'] | conversation_b roles: ['user', 'assistant']
  conversation_b extra keys: ['num_tokens']
  full_conversation turns: 1 | keys per turn: ['user', 'model_a', 'model_b']
  last turn matches conversation_a/b: True True True
row 7 (order 2):
  conversation_a roles: ['user', 'assistant'] | conversation_b roles: ['user', 'assistant']
  conversation_b extra keys: ['num_tokens']
  full_conversation turns: 2 | keys per turn: ['user', 'model_a', 'model_b']
  last turn matches conversation_a/b: True True True
row 53 (order 3):
  conversation_a roles: ['user', 'assistant'] | conversation_b roles: ['user', 'assistant']
  conversation_b extra keys: ['num_tokens']
  full_conversation turns: 3 | keys per turn: ['user', 'model_a', 'model_b']
  last turn matches conversation_a/b: True True True


In [13]:
# For each history turn of the multi-turn samples, find rows whose *current* turn has the same
# prompt and responses. Lookup only: a cheap substring pre-check, then parsing of candidates.
convs_a = train["conversation_a"]

def rows_with_current_turn(turn):
    prompt = text_of(turn["user"])
    key = repr(prompt[:150])[1:-1]
    hits = []
    for j, s in enumerate(convs_a):
        if key in s[:8000]:
            ca = parse_conversation(s)
            if text_of(ca[0]) == prompt and text_of(ca[1]) == text_of(turn["model_a"]):
                cb = parse_conversation(train[j]["conversation_b"])
                hits.append((j, train[j]["model_a"], train[j]["model_b"], train[j]["evaluation_order"],
                             text_of(cb[1]) == text_of(turn["model_b"])))
    return hits

for i in sample_idx[1:]:
    r = train[i]
    print(f"row {i}: current models {r['model_a']} vs {r['model_b']}")
    for k, turn in enumerate(parse_conversation(r["full_conversation"]), 1):
        print(f"  turn {k}: (row, model_a, model_b, order, model_b response also matches) ->", rows_with_current_turn(turn))

row 7: current models o3-2025-04-16 vs grok-3-mini-beta

  turn 1: (row, model_a, model_b, order, model_b response also matches) -> []


  turn 2: (row, model_a, model_b, order, model_b response also matches) -> [(7, 'o3-2025-04-16', 'grok-3-mini-beta', 2, True)]
row 53: current models o3-2025-04-16 vs command-a-03-2025


  turn 1: (row, model_a, model_b, order, model_b response also matches) -> [(2335, 'amazon-nova-experimental-chat-05-14', 'hunyuan-turbos-20250416', 1, True)]


  turn 2: (row, model_a, model_b, order, model_b response also matches) -> []


  turn 3: (row, model_a, model_b, order, model_b response also matches) -> [(53, 'o3-2025-04-16', 'command-a-03-2025', 3, True)]


## Dataset verification: full-dataset parse and structure checks

Run `parse_conversation` on every row of all three conversation fields. Failures are counted and shown with reasons; they are not repaired or dropped. Nothing is extracted or filtered here.

In [14]:
# Parse every row of each field; keep successes and failures separately
parsed = {col: {} for col in CONV_COLS}
failures = {col: [] for col in CONV_COLS}
for col in CONV_COLS:
    for i, s in enumerate(train[col]):
        try:
            parsed[col][i] = parse_conversation(s)
        except Exception as e:
            failures[col].append((i, f"{type(e).__name__}: {str(e)[:100]}"))
    print(f"{col}: parsed={len(parsed[col])}, failed={len(failures[col])}")
    for i, reason in failures[col][:3]:
        print(f"  row {i}: {reason}")

conversation_a: parsed=5128, failed=0


conversation_b: parsed=5128, failed=0


full_conversation: parsed=5128, failed=0


In [15]:
# Message roles, content items per message, and content-item types (current-turn fields + history)
roles, n_items, item_types = Counter(), Counter(), Counter()

def messages(col, conv):
    if col == "full_conversation":
        return [m for turn in conv for m in turn.values()]
    return conv

for col in CONV_COLS:
    for conv in parsed[col].values():
        for m in messages(col, conv):
            roles[(col, m["role"])] += 1
            if m["content"] is None:              # counted, not skipped silently
                n_items[(col, "content=None")] += 1
                continue
            n_items[(col, len(m["content"]))] += 1
            for item in m["content"]:
                item_types[(col, item["type"])] += 1
print("Roles:", dict(roles))
print("Content items per message:", dict(n_items))
print("Content item types:", dict(item_types))

Roles: {('conversation_a', 'user'): 6794, ('conversation_a', 'assistant'): 6794, ('conversation_b', 'user'): 6794, ('conversation_b', 'assistant'): 6794, ('full_conversation', 'user'): 9979, ('full_conversation', 'assistant'): 19958}
Content items per message: {('conversation_a', 1): 13576, ('conversation_a', 0): 10, ('conversation_a', 'content=None'): 2, ('conversation_b', 1): 13573, ('conversation_b', 0): 13, ('conversation_b', 'content=None'): 2, ('full_conversation', 1): 29903, ('full_conversation', 0): 27, ('full_conversation', 'content=None'): 4, ('full_conversation', 3): 2, ('full_conversation', 2): 1}
Content item types: {('conversation_a', 'text'): 13576, ('conversation_b', 'text'): 13573, ('full_conversation', 'text'): 29906, ('full_conversation', 'image'): 5}


In [16]:
# Final-pair checks on rows where all three fields parsed.
# Text is compared separately from metadata (conversation_b messages carry num_tokens; history does not).
def texts(m):
    return None if m["content"] is None else [item.get("text") for item in m["content"]]

def alternates(conv):
    return len(conv) % 2 == 0 and [m["role"] for m in conv] == ["user", "assistant"] * (len(conv) // 2)

both = sorted(set(parsed["conversation_a"]) & set(parsed["conversation_b"]) & set(parsed["full_conversation"]))
checks = {name: [] for name in ["a alternates user/assistant", "b alternates user/assistant", "final prompt a == b",
                                "final pair text == last full turn", "final pair incl. metadata == last full turn",
                                "history length == evaluation_order"]}
for i in both:
    a, b, last = parsed["conversation_a"][i], parsed["conversation_b"][i], parsed["full_conversation"][i][-1]
    results = [
        alternates(a), alternates(b),
        texts(a[-2]) == texts(b[-2]),
        texts(a[-2]) == texts(last["user"]) and texts(a[-1]) == texts(last["model_a"]) and texts(b[-1]) == texts(last["model_b"]),
        a[-2] == last["user"] and a[-1] == last["model_a"] and b[-1] == last["model_b"],
        len(parsed["full_conversation"][i]) == orders[i],
    ]
    for name, ok in zip(checks, results):
        if not ok:
            checks[name].append(i)

print(f"Rows checked: {len(both)}")
for name, bad in checks.items():
    print(f"{name}: {len(both) - len(bad)} pass, {len(bad)} fail; examples: {bad[:5]}")

Rows checked: 5128
a alternates user/assistant: 5128 pass, 0 fail; examples: []
b alternates user/assistant: 5128 pass, 0 fail; examples: []
final prompt a == b: 5128 pass, 0 fail; examples: []
final pair text == last full turn: 5128 pass, 0 fail; examples: []
final pair incl. metadata == last full turn: 0 pass, 5128 fail; examples: [0, 1, 2, 3, 4]
history length == evaluation_order: 4161 pass, 967 fail; examples: [5, 8, 17, 29, 35]


In [17]:
# Empty final prompts/responses, counted by side and reason (counted only; nothing is dropped)
def empty_reason(m):
    if m["content"] is None: return "content=None"
    if len(m["content"]) == 0: return "empty content list"
    if all(not (item.get("text") or "").strip() for item in m["content"]): return "blank text"
    return None

empties = Counter()
for side in ["conversation_a", "conversation_b"]:
    for i, conv in parsed[side].items():
        for part, m in [("prompt", conv[-2]), ("response", conv[-1])]:
            reason = empty_reason(m)
            if reason:
                empties[(side, part, reason)] += 1
for key, n in sorted(empties.items()):
    print(key, n)

('conversation_a', 'response', 'blank text') 1
('conversation_a', 'response', 'content=None') 2
('conversation_a', 'response', 'empty content list') 9
('conversation_b', 'response', 'content=None') 2
('conversation_b', 'response', 'empty content list') 10


### Multi-turn structure

Compare the number of pairs in `conversation_a` with the number of turns in `full_conversation`.

In [18]:
# Structural categories: pairs in conversation_a vs turns in full_conversation
def category(i):
    n_pairs, n_full = len(parsed["conversation_a"][i]) // 2, len(parsed["full_conversation"][i])
    if n_pairs == 1:
        return "single pair"
    return "multi pair, = full turns" if n_pairs == n_full else "multi pair, < full turns" if n_pairs < n_full else "multi pair, > full turns"

categories = {}
for i in both:
    categories.setdefault(category(i), []).append(i)
for name, rows in categories.items():
    print(f"{name}: {len(rows)} rows; evaluation_order counts: {dict(sorted(Counter(orders[i] for i in rows).items()))}")

single pair: 4323 rows; evaluation_order counts: {1: 3347, 2: 490, 3: 217, 4: 92, 5: 75, 6: 35, 7: 17, 8: 10, 9: 8, 10: 8, 11: 4, 12: 8, 13: 2, 14: 2, 15: 1, 16: 2, 17: 2, 19: 1, 21: 1, 24: 1}


multi pair, = full turns: 630 rows; evaluation_order counts: {1: 630}
multi pair, < full turns: 175 rows; evaluation_order counts: {2: 103, 3: 38, 4: 14, 5: 5, 6: 5, 7: 2, 8: 3, 9: 3, 11: 1, 17: 1}


In [19]:
# One example per category, plus final-pair text mismatches: compare prompts turn by turn
def show(i):
    a, full = parsed["conversation_a"][i], parsed["full_conversation"][i]
    print(f"row {i} | order {orders[i]} | {train[i]['model_a']} vs {train[i]['model_b']} | "
          f"pairs in a: {len(a) // 2}, full turns: {len(full)}")
    a_prompts = [texts(m)[0][:50] for m in a[::2]]
    full_prompts = [texts(t["user"])[0][:50] for t in full]
    print("  a prompts:   ", a_prompts)
    print("  full prompts:", full_prompts)

for name, rows in categories.items():
    print(f"--- {name}")
    show(rows[0])
print("--- final-pair text mismatches:", len(checks["final pair text == last full turn"]))
for i in checks["final pair text == last full turn"][:3]:
    show(i)

--- single pair
row 0 | order 1 | claude-3-5-sonnet-20241022 vs gemma-3-27b-it | pairs in a: 1, full turns: 1
  a prompts:    ['一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000']
  full prompts: ['一个长度为n的非负整数序列a，满足单调不降，且所有项的和为S，求符合条件的序列有多少个（n=1000']
--- multi pair, = full turns
row 5 | order 1 | claude-3-7-sonnet-20250219-thinking-32k vs qwen3-coder-480b-a35b-instruct | pairs in a: 4, full turns: 4
  a prompts:    ["is a good idea to set 'cron' => ['index' => ['thre", 'But at the moment there is no "cron" entry, so the', 'I experience Server overload\nMemory exhaustion\nDat', 'Magento uses the global default, which is effectiv']
  full prompts: ["is a good idea to set 'cron' => ['index' => ['thre", 'But at the moment there is no "cron" entry, so the', 'I experience Server overload\nMemory exhaustion\nDat', 'Magento uses the global default, which is effectiv']
--- multi pair, < full turns
row 79 | order 2 | llama-4-scout-17b-16e-instruct vs deepseek-v3-0324 | pairs in a: 5, full turns

## Family mapping proposal (for author review)

Counts of final responses per exact model identifier, with a **proposed** label. These are proposals, not approved mappings. Labels come from the identifier's naming pattern, not from provider ownership alone.

- **English counts** use the dataset's row-level `language == "en"` label. That label does not guarantee every final pair is entirely English.
- **All counts** are before deduplication and balancing.

In [20]:
import re
import pandas as pd

def propose(model):
    """Proposed family label and a reason for ambiguous cases."""
    if "-invalid" in model:
        return "needs review", "'-invalid' suffix; meaning undocumented"
    if model.startswith("claude-"):
        return "Claude", ""
    if model.startswith("gpt-oss-"):
        return "needs review", "open-weight OpenAI model, not a GPT chat product"
    if model.startswith(("gpt-", "chatgpt-")):
        return "GPT", ""
    if re.match(r"o\d", model):
        return "needs review", "OpenAI o-series reasoning model, not named GPT"
    if model.startswith("qwq-"):
        return "needs review", "Qwen-team reasoning model, not named qwen-*"
    if model.startswith(("qwen-", "qwen3-")) and "-vl-" in model:
        return "needs review", "Qwen vision-language variant"
    if model.startswith(("qwen-", "qwen3-")):
        return "Qwen", ""
    return "excluded", ""

# Count final responses per model: total, non-empty, and non-empty in "en" rows
counts = Counter()
for side, model_col in [("conversation_a", "model_a"), ("conversation_b", "model_b")]:
    models = train[model_col]
    for i, conv in parsed[side].items():
        m = models[i]
        counts[(m, "total")] += 1
        if empty_reason(conv[-1]) is None:
            counts[(m, "non-empty")] += 1
            if langs[i] == "en":
                counts[(m, "non-empty en")] += 1

model_table = pd.DataFrame(
    [{"model": m, "total": counts[(m, "total")], "non-empty": counts[(m, "non-empty")],
      "non-empty en": counts[(m, "non-empty en")], "proposed": propose(m)[0], "reason": propose(m)[1]}
     for m in sorted({m for m, _ in counts})]
).sort_values(["proposed", "model"]).reset_index(drop=True)

In [21]:
# Proposed included and needs-review identifiers
shown = model_table[model_table["proposed"] != "excluded"]
pd.set_option("display.max_rows", 100)
shown

,model,total,non-empty,non-empty en,proposed,reason
0,claude-3-5-haiku-20241022,210,210,124,Claude,
1,claude-3-5-sonnet-20241022,194,194,130,Claude,
2,claude-3-7-sonnet-20250219,169,169,105,Claude,
3,claude-3-7-sonnet-20250219-thinking-32k,169,169,98,Claude,
4,claude-opus-4-1-20250805,179,179,108,Claude,
5,claude-opus-4-1-20250805-thinking-16k,113,113,63,Claude,
6,claude-opus-4-1-20250805-thinking-16k-old,2,2,2,Claude,
7,claude-opus-4-20250514,246,246,145,Claude,
8,claude-opus-4-20250514-thinking-16k,183,183,109,Claude,
9,claude-sonnet-4-20250514,243,243,129,Claude,


In [22]:
# Totals per proposed label, and a short summary of excluded identifiers
print(model_table.groupby("proposed")[["total", "non-empty", "non-empty en"]].sum())
excluded = model_table[model_table["proposed"] == "excluded"]
print()
print(f"Excluded: {len(excluded)} identifiers, {excluded['total'].sum()} final responses. Name prefixes:")
print(dict(Counter(re.split(r"[-.]", m)[0].lower() for m in excluded["model"]).most_common()))

              total  non-empty  non-empty en
proposed                                    
Claude         1931       1931          1138
GPT            1109       1109           655
Qwen           1351       1349           774
excluded       4805       4797          2856
needs review   1060       1046           631

Excluded: 57 identifiers, 4805 final responses. Name prefixes:
{'deepseek': 8, 'gemini': 8, 'grok': 7, 'glm': 4, 'llama': 4, 'mistral': 4, 'hunyuan': 3, 'mimo': 2, 'amazon': 2, 'gemma': 2, 'kimi': 2, 'step': 2, 'command': 1, 'ibm': 1, 'ling': 1, 'longcat': 1, 'magistral': 1, 'mai': 1, 'minimax': 1, 'nvidia': 1, 'ring': 1}
